In [ ]:
api_key = "####YOUR_API_KEY####"

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn.manifold import TSNE
from tqdm import tqdm
import plotly.express as px
import requests
import time
import json

In [ ]:
newsdata = pd.read_csv('path/to/real-world-nonparallel.csv')

In [ ]:
newsdata.head(2)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
tqdm.pandas()

# language codes map : https://cloud.google.com/translate/docs/languages
lang_map = {
    "meitei": "mni-Mtei", "oriya": "or", "hindi": "hi", "tamil": "ta", "bengali": "bn", "urdu": "ur",
    "assamese": "as", "marathi": "mr", "telugu": "te", "gujarati": "gu", "punjabi": "pa",
    "kannada": "kn", "malayalam": "ml"
}

# Translation using Google Cloud Translation API
def translate_to_english(text, source_lang_code, api_key):
    try:
        if not isinstance(text, str) or text.strip() == "" or source_lang_code == "en":
            return text

        url = "https://translation.googleapis.com/language/translate/v2"
        params = {
            "q": text,
            "source": source_lang_code,
            "target": "en",
            "key": api_key,
            "format": "text"
        }

        response = requests.post(url, data=params)
        if response.status_code == 200:
            return response.json()["data"]["translations"][0]["translatedText"]
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return text
    except Exception as e:
        print(f"Translation error from {source_lang_code}: {text[:50]}..., Error: {e}")
        return text

In [ ]:
def translate(row, api_key):
    lang = row['language']
    text = row['title']
    if lang != 'english' and lang in lang_map:
        return translate_to_english(text, lang_map[lang], api_key)
    return text


In [ ]:
newsdata_translated = newsdata.copy()
newsdata_translated['translated_title'] = newsdata.progress_apply(lambda row: translate(row, api_key), axis=1)

`newsdata_translated.csv` is saved in the repository and can help to avoid redoing the translations.

In [ ]:
newsdata_translated.head()

# Generate Embeddings using MPNet

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model = model.to(device)

In [ ]:
# Generate embeddings
translated_title_embeddings = []
with tqdm(total=len(newsdata_translated), desc="Generating Title Embeddings", dynamic_ncols=True, leave=True) as pbar:
    for title in newsdata_translated["translated_title"].astype(str):
        embedding = model.encode(title, convert_to_tensor=True, device=device).cpu().numpy()  # Move to CPU
        translated_title_embeddings.append(embedding.tolist())  # Convert NumPy array to list for JSON storage
        pbar.update(1)

# Add embeddings to DataFrame
newsdata_translated["translated_title_embeddings"] = translated_title_embeddings

# do dimensionality reduction using t-SNE

In [ ]:
embeddings = np.vstack(newsdata_translated["translated_title_embeddings"].values)

# Run t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=50,
    max_iter=2000,
    learning_rate=200,
    metric="cosine",
    random_state=42
)

tsne_results = tsne.fit_transform(embeddings)

## Semantic Representation Map Coloured by Language

In [ ]:
newsdata_translated["x"] = tsne_results[:, 0]
newsdata_translated["y"] = tsne_results[:, 1]

# Plot
fig = px.scatter(
    newsdata_translated, x="x", y="y", color="language",
    hover_data=["title", "translated_title", "source_name", "language"],
    title="t-SNE Clustering of News Articles"
)
fig.show()